In [ ]:
!pip install transformers
!pip install torch
!pip install scikit-learn
!pip install gradio
!pip install nltk

In [ ]:
import pandas as pd

fake = pd.read_csv("Fake.csv")
true = pd.read_csv("True.csv")

fake["label"] = 0
true["label"] = 1

df = pd.concat([fake, true], axis=0)
df = df.sample(frac=1).reset_index(drop=True)

df.head()

In [ ]:
df["content"] = df["title"] + " " + df["text"]

df = df[["content", "label"]]
df.head()

In [ ]:
df["label"].value_counts()

In [ ]:
import matplotlib.pyplot as plt

df["label"].value_counts().plot(kind="bar")

plt.title("Class Distribution")
plt.xticks([0,1], ["Fake", "Real"], rotation=0)
plt.show()

In [ ]:
import nltk
import re

nltk.download("stopwords")

from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

def clean_text(text):

    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z]", " ", text)

    words = text.split()
    words = [word for word in words if word not in stop_words]

    return " ".join(words)

df["content"] = df["content"].apply(clean_text)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["content"],
    df["label"],
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [ ]:
from sklearn.svm import LinearSVC

svm_model = LinearSVC()
svm_model.fit(X_train_tfidf, y_train)

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    eval_metric="logloss",
    use_label_encoder=False
)

xgb_model.fit(X_train_tfidf, y_train)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier()
rf_model.fit(X_train_tfidf, y_train)

In [ ]:
from sklearn.metrics import accuracy_score

def evaluate(model, X_test, y_test):

    pred = model.predict(X_test)

    print("Accuracy:", accuracy_score(y_test, pred))

evaluate(svm_model, X_test_tfidf, y_test)
evaluate(rf_model, X_test_tfidf, y_test)
evaluate(xgb_model, X_test_tfidf, y_test)

In [ ]:
from transformers import DistilBertTokenizer, DistilBertModel
import torch

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
model = DistilBertModel.from_pretrained("distilbert-base-uncased")

In [ ]:
import numpy as np

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


def get_embeddings(text_list, batch_size=32):

    embeddings = []

    for i in range(0, len(text_list), batch_size):

        batch = text_list[i:i+batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=256,   # faster than 512 and still accurate
            return_tensors="pt"
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():

            outputs = model(**inputs)

        cls_embeddings = outputs.last_hidden_state[:, 0, :]

        embeddings.append(cls_embeddings.cpu().numpy())

    return np.vstack(embeddings)

In [ ]:
X_train_emb = get_embeddings(X_train.tolist())
X_test_emb = get_embeddings(X_test.tolist())

In [ ]:
svm_hybrid = LinearSVC()
svm_hybrid.fit(X_train_emb, y_train[:len(X_train_emb)])

In [ ]:
rf_hybrid = RandomForestClassifier()

rf_hybrid.fit(X_train_emb, y_train[:len(X_train_emb)])

In [ ]:
xgb_hybrid = XGBClassifier()

xgb_hybrid.fit(X_train_emb, y_train[:len(X_train_emb)])

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import time


def evaluate_model(model, X_test, y_test, model_name):

    print(f"\n===== {model_name} Evaluation =====")

    # Inference latency
    start = time.time()
    predictions = model.predict(X_test)
    end = time.time()

    latency = end - start

    # Accuracy + Precision + Recall + F1
    print("\nClassification Report:\n")
    print(classification_report(y_test, predictions))

    # Confusion Matrix
    cm = confusion_matrix(y_test, predictions)

    disp = ConfusionMatrixDisplay(confusion_matrix=cm)

    disp.plot(cmap="Blues")

    plt.title(f"{model_name} Confusion Matrix")

    plt.show()

    # Latency output
    print(f"Inference Latency: {latency:.6f} seconds")

In [ ]:
evaluate_model(
    svm_hybrid,
    X_test_emb,
    y_test[:len(X_test_emb)],
    "DistilBERT + SVM Hybrid Model"
)

evaluate_model(
    rf_hybrid,
    X_test_emb,
    y_test[:len(X_test_emb)],
    "DistilBERT + Random Forest Hybrid Model"
)

evaluate_model(
    xgb_hybrid,
    X_test_emb,
    y_test[:len(X_test_emb)],
    "DistilBERT + XGBoost Hybrid Model"
)

In [ ]:
import pickle

pickle.dump(svm_hybrid, open("best_model.pkl","wb"))
pickle.dump(tfidf, open("tfidf.pkl","wb"))

In [ ]:
import gradio as gr

In [ ]:
def predict_news(text):

    cleaned_text = clean_text(text)

    embedding = get_embeddings([cleaned_text])

    prediction = svm_hybrid.predict(embedding)[0]

    # confidence score (approximate)
    confidence = np.max(svm_hybrid.decision_function(embedding))

    if prediction == 1:

        result = " REAL NEWS"

    else:

        result = " FAKE NEWS"

    return result, f"Confidence Score: {confidence:.2f}"


with gr.Blocks(theme=gr.themes.Soft()) as interface:

    gr.Markdown(
        """
        # Hybrid AI Fake News Detection System
        ### DistilBERT + Machine Learning Model

        Paste any news article text below to check whether it is **Real or Fake**.
        """
    )

    with gr.Row():

        with gr.Column():

            news_input = gr.Textbox(
                label="Enter News Article Text",
                placeholder="Paste news content here...",
                lines=10
            )

            check_button = gr.Button(" Detect News Authenticity")

        with gr.Column():

            prediction_output = gr.Textbox(label="Prediction Result")

            confidence_output = gr.Textbox(label="Model Confidence")


    check_button.click(
        fn=predict_news,
        inputs=news_input,
        outputs=[prediction_output, confidence_output]
    )


    gr.Markdown(
        """
        ---
        Model Architecture:
        - DistilBERT contextual embeddings
        - Hybrid classifier
        - TF-IDF linguistic features

        Developed for academic research deployment demonstration
        """
    )


interface.launch()